# Demo rápida de IOM: reglas, CSV y resultados

Este recorrido usa artefactos generados incluidos en el repositorio y **no compila Lean**. No vuelve a entrenar modelos ni valida externamente la teoría; inspecciona el gold de reglas y los resultados reproducibles guardados.

In [ ]:
from pathlib import Path
import csv, json
ROOT = Path.cwd()
if not (ROOT / 'datasets/relation_candidates.csv').exists():
    ROOT = ROOT.parent
assert (ROOT / 'datasets/relation_candidates.csv').exists(), 'Abre el notebook desde la raíz del repo'
print('Repo:', ROOT)

## 1. Cargar artefactos precompilados
`triad_nodes.csv` describe nodos y `relation_candidates.csv` incluye etiquetas derivadas de RDF. El notebook evita compilación Lean y entrenamiento.

In [ ]:
def read_csv(name):
    with (ROOT / 'datasets' / name).open(encoding='utf-8', newline='') as f:
        return list(csv.DictReader(f))
nodes = read_csv('triad_nodes.csv')
relations = read_csv('relation_candidates.csv')
print(f'{len(nodes)} nodos; {len(relations)} candidatos')
print('Distribución:', {label: sum(r['relation'] == label for r in relations) for label in ('none','mirrorOf','creates')})

## 2. Inspeccionar una relación y su estructura
El ejemplo se consulta del dataset canónico; la derivación formal completa es verificable ejecutando `python scripts/verify_bridge.py`.

In [ ]:
positive = next(r for r in relations if r['relation'] == 'mirrorOf')
print(positive)
by_id = {n['node_id']: n for n in nodes}
print('origen:', by_id[positive['source']])
print('destino:', by_id[positive['target']])

## 3. Consultar PI-HGAT-T y baselines
Las cifras siguientes corresponden a los resultados guardados; para reproducir entrenamiento, sigue `EXPERIMENTOS.md`.

In [ ]:
pi = json.loads((ROOT / 'experiments/results/pi_hgat_t_metrics.json').read_text())
baseline = json.loads((ROOT / 'experiments/results/baselines_comparison.json').read_text())
print('Validación:', pi['validation'])
for exp in pi['experiments']:
    m = exp['metrics']
    print(exp['variant'], 'accuracy=', m['candidate_relation_accuracy']['value'], 'macro-F1=', m['relation_macro_f1']['value'])
for name, result in baseline['baselines'].items():
    print(name, 'accuracy=', result['metrics']['accuracy'], 'macro-F1=', result['metrics']['macro_f1'])

## Interpretación
Las métricas estiman recuperabilidad de etiquetas generadas por reglas bajo partición leave-one-triad-out. No son validación independiente del marco. Para una revisión autónoma de humanos, consulta `datasets/independent_eval/PROTOCOL.json` y verifica primero su estado de ejecución.